# Step 4 – TTS (normalised Bangla text): অনুবাদ থেকে বাংলা আর ইংরেজি কণ্ঠ (MMS-TTS)
**Runtime → Change runtime type → T4 GPU**, তারপর **Run all**। GPU না থাকলেও চলে, শুধু ধীর।
কোনো ট্রেনিং নেই: Meta-র আগে থেকে তৈরি `facebook/mms-tts-ben` আর `facebook/mms-tts-eng` ব্যবহার হয়।
মূল্যায়ন: MT-র ১০০টা অনুবাদ কণ্ঠে বানিয়ে আবার Whisper দিয়ে লেখা হয় (বোধগম্যতা = CER/WER), সাথে গতি (RTF)।
ফল সেভ হয় Drive এর `sylheti_tts` ফোল্ডারে।

**এই সংস্করণে:** বাংলা লেখা TTS-এ পাঠানোর আগে পরিষ্কার করা হয় (সংখ্যা → কথা, যতিচিহ্ন), আর মূল্যায়নে Whisper-এর হিন্দি (দেবনাগরী) অক্ষর বাংলায় বদলে আলাদা CER দেখানো হয়। ফল যায় `sylheti_tts/run_normalized/` ফোল্ডারে — আগের ফল বদলায় না।

## 1. Drive আর library

In [ ]:
from google.colab import drive
import os, sys, subprocess
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
N = 100              # how many test sentences to synthesise and evaluate
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.48.3", "accelerate==1.3.0",
                "jiwer==3.0.5", "soundfile", "uroman"], check=True)
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if dev == "cuda" else "CPU (slow but works)")

## 2. MT-র অনুবাদ খুঁজে আনা

In [ ]:
import pandas as pd
D = "/content/drive/MyDrive/Sylheti_Thesis"
cands = [D + "/sylheti_mt_copy/nllb-sylheti", D + "/sylheti_mt/nllb-sylheti"]
MT_RUN = next((p for p in cands if os.path.exists(p + "/predictions_test.csv")), None)
assert MT_RUN, "predictions_test.csv পাওয়া যায়নি: sylheti_mt (বা sylheti_mt_copy) My Drive-এ আছে কিনা দেখুন"
print("MT results from:", MT_RUN)
pred = pd.read_csv(MT_RUN + "/predictions_test.csv", encoding="utf-8-sig")
pred = pred.dropna(subset=["pred_bn", "pred_en"]).sample(min(N, len(pred)), random_state=13).reset_index(drop=True)
OUT = D + "/sylheti_tts"; os.makedirs(OUT + "/wav_bn", exist_ok=True); os.makedirs(OUT + "/wav_en", exist_ok=True)
print(len(pred), "sentences | example:", pred.pred_bn[0], "|", pred.pred_en[0])

## 2b. বাংলা লেখা পরিষ্কার করা (TTS-এর আগে)

In [ ]:
# ===== বাংলা লেখা পরিষ্কার করা (TTS-এ পাঠানোর আগে) =====
import re, unicodedata

_ONES = ("শূন্য এক দুই তিন চার পাঁচ ছয় সাত আট নয় দশ এগারো বারো তেরো চোদ্দ পনেরো ষোলো সতেরো আঠারো উনিশ "
         "বিশ একুশ বাইশ তেইশ চব্বিশ পঁচিশ ছাব্বিশ সাতাশ আটাশ উনত্রিশ ত্রিশ একত্রিশ বত্রিশ তেত্রিশ চৌত্রিশ পঁয়ত্রিশ ছত্রিশ সাঁইত্রিশ আটত্রিশ উনচল্লিশ "
         "চল্লিশ একচল্লিশ বিয়াল্লিশ তেতাল্লিশ চুয়াল্লিশ পঁয়তাল্লিশ ছেচল্লিশ সাতচল্লিশ আটচল্লিশ উনপঞ্চাশ পঞ্চাশ একান্ন বাহান্ন তিপ্পান্ন চুয়ান্ন পঞ্চান্ন ছাপ্পান্ন সাতান্ন আটান্ন উনষাট "
         "ষাট একষট্টি বাষট্টি তেষট্টি চৌষট্টি পঁয়ষট্টি ছেষট্টি সাতষট্টি আটষট্টি উনসত্তর সত্তর একাত্তর বাহাত্তর তিয়াত্তর চুয়াত্তর পঁচাত্তর ছিয়াত্তর সাতাত্তর আটাত্তর উনআশি "
         "আশি একাশি বিরাশি তিরাশি চুরাশি পঁচাশি ছিয়াশি সাতাশি আটাশি উননব্বই নব্বই একানব্বই বিরানব্বই তিরানব্বই চুরানব্বই পঁচানব্বই ছিয়ানব্বই সাতানব্বই আটানব্বই নিরানব্বই").split()
assert len(_ONES) == 100
_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def num_to_bn(n: int) -> str:
    """পূর্ণ সংখ্যা → বাংলা কথায় (কোটি পর্যন্ত, তার বেশি হলে কোটি-র আগে আবার একই নিয়ম)"""
    if n < 100: return _ONES[n]
    parts = []
    for div, word in ((10**7, "কোটি"), (10**5, "লাখ"), (1000, "হাজার")):
        if n >= div:
            parts.append(num_to_bn(n // div) + " " + word); n %= div
    if n >= 100:
        parts.append(_ONES[n // 100] + "শো"); n %= 100
    if n: parts.append(_ONES[n])
    return " ".join(parts)

def _number(m):
    s = m.group(0).translate(_BN_DIGITS).replace(",", "")
    if "." in s:
        a, b = s.split(".", 1)
        return num_to_bn(int(a or 0)) + " দশমিক " + " ".join(_ONES[int(d)] for d in b)
    return num_to_bn(int(s))

def normalize_bn(text: str) -> str:
    t = unicodedata.normalize("NFC", str(text))
    t = t.replace("\u200c", "").replace("\u200d", "")                      # ZWNJ / ZWJ
    t = re.sub(r"[০-৯0-9]+(?:[,][০-৯0-9]{2,3})*(?:\.[০-৯0-9]+)?", _number, t)  # সংখ্যা → কথা
    t = t.replace("%", " শতাংশ").replace("&", " এবং ")
    t = re.sub(r"[/|\\]", " অথবা ", t)                                       # দিন/সপ্তাহ → দিন অথবা সপ্তাহ
    t = re.sub(r"[\"'“”‘’«»()\[\]{}<>*_#~^]", " ", t)                        # উদ্ধৃতি, বন্ধনী
    t = re.sub(r"\s*[-–—]\s*", " ", t)                                       # ড্যাশ → ফাঁকা
    t = re.sub(r"\.{2,}|…", "।", t)                                          # ... → দাঁড়ি
    t = re.sub(r"(?<=[\u0980-\u09FF])\.(?=\s|$)", "।", t)                    # বাংলা শব্দের পরে . → ।
    t = re.sub(r"\s*([।?!,;:])\s*", r"\1 ", t)                               # যতিচিহ্নের আগে ফাঁকা নয়
    t = re.sub(r"([।?!])[।?!\s]*", r"\1 ", t)                                # একাধিক দাঁড়ি → একটা
    t = re.sub(r"\s+", " ", t).strip()
    if t and t[-1] not in "।?!": t += "।"                                    # শেষে দাঁড়ি
    return t

def has_latin(text: str) -> bool:
    """ইংরেজি অক্ষর থাকলে True — বাংলা TTS এগুলো ঠিকমতো পড়তে পারে না"""
    return bool(re.search(r"[A-Za-z]", str(text)))

# ---- পরীক্ষা ----
for s in ["আমি এখানে ৩ দিন/সপ্তাহের জন্য আছি", "দাম ১,২৫০ টাকা (প্রায়)...", "২০২৪ সালে ৯৫% মানুষ এসেছিল",
          "তাপমাত্রা ৩৬.৫ ডিগ্রি", "সে বলল, “আমি যাব”", "Ok, আমি আসছি"]:
    print(f"{s}\n  → {normalize_bn(s)}" + ("   ⚠ ইংরেজি অক্ষর আছে" if has_latin(s) else ""))


## 3. কণ্ঠ বানানো (MMS-TTS)

In [ ]:
import time, re, numpy as np, soundfile as sf
from transformers import VitsModel, AutoTokenizer
try:
    import uroman as ur; _uroman = ur.Uroman()
except Exception:
    _uroman = None
tts = {}
for k, name in [("bn", "facebook/mms-tts-ben"), ("en", "facebook/mms-tts-eng")]:
    tts[k] = (AutoTokenizer.from_pretrained(name), VitsModel.from_pretrained(name).to(dev).eval())

# new results go to a sub-folder; the earlier run in sylheti_tts/ is kept as it is
OUT = D + "/sylheti_tts/run_normalized"
os.makedirs(OUT + "/wav_bn", exist_ok=True); os.makedirs(OUT + "/wav_en", exist_ok=True)

def speak(text, lang, seed=0):
    tok, m = tts[lang]
    text = re.sub(r"\s+", " ", str(text)).strip()
    if getattr(tok, "is_uroman", False) and _uroman is not None:
        text = _uroman.romanize_string(text)
    if lang == "en":
        text = text.lower()
    ids = tok(text, return_tensors="pt").to(dev)
    torch.manual_seed(seed)                      # same text -> same voice every time
    with torch.no_grad():
        wav = m(**ids).waveform[0].cpu().numpy()
    return wav, m.config.sampling_rate

rows = []
for i, r in pred.iterrows():
    for k in ["bn", "en"]:
        text = r["pred_" + k]
        text_tts = normalize_bn(text) if k == "bn" else text      # Bangla: numbers -> words, clean punctuation
        t0 = time.time(); wav, sr = speak(text_tts, k); dt = time.time() - t0
        path = f"{OUT}/wav_{k}/{i:03d}.wav"; sf.write(path, wav, sr)
        rows.append({"id": i, "lang": k, "text": text, "text_tts": text_tts, "changed": text_tts != text,
                     "has_latin": has_latin(text) if k == "bn" else False,
                     "wav": path, "sec": len(wav) / sr, "gen_sec": dt})
tt = pd.DataFrame(rows); tt.to_csv(OUT + "/tts_outputs.csv", index=False, encoding="utf-8-sig")
b = tt[tt.lang == "bn"]
print(f"bn: normalisation changed {b.changed.sum()} of {len(b)} sentences; {b.has_latin.sum()} contain English letters")
for _, r in b[b.changed].head(5).iterrows():
    print("  ", r.text, "\n   →", r.text_tts)
for k in ["bn", "en"]:
    s = tt[tt.lang == k]
    print(f"{k}: {len(s)} clips, {s.sec.sum()/60:.1f} min audio, RTF {s.gen_sec.sum()/s.sec.sum():.3f} (lower = faster)")

## 4. মূল্যায়ন: তৈরি কণ্ঠ আবার Whisper দিয়ে লেখা (বোধগম্যতা)

In [ ]:
import jiwer, unicodedata, librosa
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               torch_dtype=torch.float16 if dev == "cuda" else torch.float32, device=0 if dev == "cuda" else -1)
# clear the model's default decoder prompt so only the language/task given below are used
asr.model.generation_config.forced_decoder_ids = None

def norm(s, lang):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
def has_deva(s):
    # Devanagari letters only; U+0964/U+0965 (danda) are also used in normal Bangla text, so they are skipped
    return bool(re.search(r"[\u0900-\u0963\u0966-\u097F]", str(s)))
def deva_to_beng(s):
    """Whisper sometimes writes Bangla audio in Hindi (Devanagari) letters; map them to the same Bengali letters"""
    out = []
    for ch in str(s):
        c = ord(ch)
        if 0x0900 <= c <= 0x097F:
            b = chr(c + 0x80)
            out.append(b if unicodedata.name(b, "") else ch)
        else:
            out.append(ch)
    return "".join(out)

res = []
for k, lang in [("bn", "bengali"), ("en", "english")]:
    s = tt[tt.lang == k].reset_index(drop=True)
    audio = [librosa.load(p, sr=16000)[0] for p in s.wav]
    out = asr(audio, batch_size=8,
              generate_kwargs={"language": lang, "task": "transcribe", "max_new_tokens": 440})
    raw = [o["text"] for o in out]
    hyp = [norm(h, k) or "-" for h in raw]
    ref = [norm(t, k) for t in s.text_tts]                      # reference = the text that was actually spoken
    tt.loc[tt.lang == k, "asr_raw"] = raw
    tt.loc[tt.lang == k, "asr_back"] = hyp
    row = {"language": k, "clips": len(s), "CER %": round(100 * jiwer.cer(ref, hyp), 1),
           "WER %": round(100 * jiwer.wer(ref, hyp), 1),
           "audio min": round(s.sec.sum() / 60, 1), "RTF": round(s.gen_sec.sum() / s.sec.sum(), 3)}
    if k == "bn":
        hyp2 = [norm(deva_to_beng(h), k) or "-" for h in raw]
        tt.loc[tt.lang == k, "asr_back_beng"] = hyp2
        row["clips with Hindi letters"] = sum(has_deva(h) for h in raw)
        row["CER % (Hindi letters→Bengali)"] = round(100 * jiwer.cer(ref, hyp2), 1)
        row["WER % (Hindi letters→Bengali)"] = round(100 * jiwer.wer(ref, hyp2), 1)
    res.append(row)

res = pd.DataFrame(res); res.to_csv(OUT + "/results.csv", index=False)
tt.to_csv(OUT + "/tts_outputs.csv", index=False, encoding="utf-8-sig")
print(res.set_index("language").T.to_string())                # vertical table: nothing gets cut off
print("\nsaved:", OUT + "/results.csv")

b = tt[(tt.lang == "bn") & tt.asr_raw.map(has_deva)].head(3)
for _, r in b.iterrows():
    print("\nলেখা     :", r.text_tts, "\nWhisper  :", r.asr_raw)

## 5. শুনে দেখুন

In [ ]:
from IPython.display import Audio, display
import random
ids = random.sample(sorted(tt.id.unique()), 3)   # new random sentences every run
for i in ids:
    print(f"---- বাক্য #{i} ----")
    for k in ["bn", "en"]:
        r = tt[(tt.id == i) & (tt.lang == k)].iloc[0]
        print(k, "| লেখা:", r.text)
        print("   | Whisper শুনেছে:", r.asr_back)
        display(Audio(r.wav))
    print()

## 6. নিজে লিখে শুনুন

In [ ]:
text_bn = "আমি এখন বাড়ি যাব"     # change these
text_en = "I will go home now"
for t, k in [(text_bn, "bn"), (text_en, "en")]:
    wav, sr = speak(t, k); print(k, t); display(Audio(wav, rate=sr))